In [2]:
import sys
sys.path.append("..")

In [2]:
from src.losses import gram_matrix

In [3]:
import torch
features = torch.tensor([[[[1., 2., 0.]], [[0., 1., 3.]]]]) # Shape: (1,2,1,3)
gram = gram_matrix(features)
print(gram)

tensor([[ 5.,  2.],
        [ 2., 10.]])
tensor([[0.8333, 0.3333],
        [0.3333, 1.6667]])


In [9]:
import torch
from src.losses import gram_matrix

g1 = gram_matrix(torch.randn(1,64,96,96))
g2 = gram_matrix(torch.randn(1,64,40,120))

print(g1.shape, g2.shape)

torch.Size([64, 64]) torch.Size([64, 64])


In [10]:
print(torch.allclose(g1,g2.t()))

False


In [7]:
f = torch.randn(1, 64, 32, 32)
flat = f.reshape(1, 64, -1)                    # [1, 64, 1024]
perm = torch.randperm(flat.shape[-1])
shuffled = flat[:, :, perm].reshape(1, 64, 32, 32)
print(f"{torch.allclose(gram_matrix(f), gram_matrix(shuffled), atol=1e-5)}")

True


In [8]:
from src.vgg import VGGFeatures
from src.image_utils import load_image

device="cuda"
extractor = VGGFeatures().to(device)
style = load_image("../data/style/starry_night.jpg", 384, device)
with torch.no_grad():
    feats = extractor(style)
for name in ["conv1_1", "conv2_1", "conv3_1", "conv4_1", "conv5_1"]:
    print("test 5:", name, feats[name].shape, "->", gram_matrix(feats[name]).shape)

test 5: conv1_1 torch.Size([1, 64, 304, 384]) -> torch.Size([64, 64])
test 5: conv2_1 torch.Size([1, 128, 152, 192]) -> torch.Size([128, 128])
test 5: conv3_1 torch.Size([1, 256, 76, 96]) -> torch.Size([256, 256])
test 5: conv4_1 torch.Size([1, 512, 38, 48]) -> torch.Size([512, 512])
test 5: conv5_1 torch.Size([1, 512, 19, 24]) -> torch.Size([512, 512])


In [11]:
g = gram_matrix(torch.randn(1, 64, 96, 96))
print("shape:", g.shape)
print("largest value:", g.abs().max().item())
print("largest asymmetry:", (g - g.t()).abs().max().item())
print("allclose with looser tolerance:", torch.allclose(g, g.t(), atol=1e-6))

shape: torch.Size([64, 64])
largest value: 0.016354164108633995
largest asymmetry: 0.0
allclose with looser tolerance: True


In [1]:
import sys
sys.path.append("..")

In [2]:
import torch
import torch.nn.functional as F
from src.vgg import VGGFeatures
from src.image_utils import load_image
from src.losses import gram_matrix, style_grams, style_loss

In [3]:
device = "cuda"
extractor = VGGFeatures().to(device)
layers = ["conv1_1", "conv2_1", "conv3_1", "conv4_1", "conv5_1"]

style = load_image("../data/style/starry_night.jpg", 384, device)
content = load_image("../data/content/Abyssinian_24.jpg", 384, device)

In [4]:
with torch.no_grad():
    style_feats = extractor(style)
    target_grams = {k: v.detach() for k, v in style_grams(style_feats).items()}

In [5]:
print("test 1:", style_loss(style_feats, target_grams))

test 1: tensor(0., device='cuda:0')


In [6]:
print("style:", style.shape, "| content:", content.shape)
with torch.no_grad():
    content_feats = extractor(content)
print("test 2:", style_loss(content_feats, target_grams))

style: torch.Size([1, 3, 304, 384]) | content: torch.Size([1, 3, 288, 384])
test 2: tensor(0.0483, device='cuda:0')


In [7]:
image = content.clone().requires_grad_(True)
loss = style_loss(extractor(image), target_grams)
loss.backward()
print("test 3: grad sum =", image.grad.abs().sum().item())

test 3: grad sum = 2.3330578804016113


In [8]:
for layer in layers:
    weights = {k: (1.0 if k == layer else 0.0) for k in layers}
    with torch.no_grad():
        got = style_loss(content_feats, target_grams, weights).item()
        expected = F.mse_loss(gram_matrix(content_feats[layer]), target_grams[layer]).item()
    print("test 4:", layer, round(got, 8), round(expected, 8), abs(got - expected) < 1e-7)

test 4: conv1_1 6.971e-05 6.971e-05 True
test 4: conv2_1 0.00734301 0.00734301 True
test 4: conv3_1 0.01347396 0.01347396 True
test 4: conv4_1 0.02586984 0.02586984 True
test 4: conv5_1 0.00155526 0.00155526 True
